# Lecture 12 — Run VLANeXt on LIBERO on the free T4

**First:** Runtime → Change runtime type → **T4 GPU**. If you are on Kaggle: Settings → Accelerator → **GPU T4 x2** (29 GB RAM; safer than Colab's 12.7 GB).

Flow: (1) verify the hardware → (2) system libraries → (3) install VLANeXt + LIBERO → (4) T4 patches → (5) download the checkpoint (16.4 GB) → (6) eval on 2–3 episodes → (7) read the result, watch the video, produce a `runs.csv` row.

Total time (hypothesis, T4): install ~10 min + download ~5 min + 3 episodes ~5 min. Run the cells in order; if you get an error, paste the **whole** cell output into the chat.

In [ ]:
# 1) Hardware check — compute capability 7.5 = T4 (NO FlashAttention-2, NO hardware bf16 support → patches needed)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch, psutil, shutil
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| GPU", torch.cuda.get_device_name(0), "| cc", torch.cuda.get_device_capability(0))
print("RAM %.1f GB | disk free %.0f GB" % (psutil.virtual_memory().total/1e9, shutil.disk_usage("/").free/1e9))
print("bf16 (including emulation):", torch.cuda.is_bf16_supported())

In [ ]:
# 2) System libraries MuJoCo needs for headless (EGL) rendering — VLANeXt COMMON_ISSUES #1
!apt-get -qq update && apt-get -qq install -y libgl1-mesa-dev libegl1-mesa-dev libxrandr2 libxcursor1 libxinerama1 libxrender1 > /dev/null
import os; os.environ["MUJOCO_GL"] = "egl"; os.environ["PYOPENGL_PLATFORM"] = "egl"
print("ok")

In [ ]:
# 3) VLANeXt + LIBERO. We install what the eval needs, NOT the whole requirements.txt
#    (no torch 2.4 / flash-attn / deepspeed / dm-reverb). mujoco 3.8.1 is VLANeXt's own pin; it works with robosuite 1.4.1 on Linux.
%cd /content
!git clone -q https://github.com/DravenALG/VLANeXt && echo "clone ok"
%cd /content/VLANeXt
!mkdir -p third_party && git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git third_party/LIBERO
!pip install -q "transformers==5.5.0" "diffusers==0.36.0" accelerate huggingface_hub sentencepiece h5py opencv-python "imageio[ffmpeg]" \
    "robosuite==1.4.1" "mujoco==3.8.1" bddl easydict cloudpickle gym pyyaml wandb tqdm tensorflow-datasets rlds 2>&1 | tail -2
!pip install -q ./third_party/LIBERO 2>&1 | tail -1
# On first import LIBERO asks "custom path? (Y/N)"; we feed the N answer through a pipe
!printf 'N\n' | python -c "import libero.libero; print('LIBERO ok')"

In [ ]:
# 4) T4 patches: flash_attention_2 → sdpa, torch.load mmap=True, VLANEXT_TASK_IDS task filter
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/vlanext_t4_patch.py
!curl -s -o config/t4_bench_config.yaml https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/t4_bench_config.yaml
!python vlanext_t4_patch.py /content/VLANeXt

In [ ]:
# 5) Checkpoint (16.4 GB; trained for the Spatial suite). The Qwen3-VL-2B backbone downloads automatically on the first model load (~4.5 GB).
!hf download DravenALG/VLANeXt VLANeXt_libero_spatial.pt --local-dir checkpoints
!ls -lh checkpoints

In [ ]:
# 6) Eval: only tasks 0, 2 and 8 (between / table center / next to the plate), 1 episode per task.
#    The first model load takes a few minutes (Qwen download + copying the 16.4 GB state_dict).
import os
os.environ["VLANEXT_TASK_IDS"] = "0,2,8"
os.environ["PYTHONPATH"] = "/content/VLANeXt/third_party/LIBERO"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"
os.environ.pop("LIBERO_CONFIG_PATH", None)     # COMMON_ISSUES #6
!CUDA_VISIBLE_DEVICES=0 python -m scripts.libero_bench_eval --config config/t4_bench_config.yaml 2>&1 | grep -v Warning | tee eval_stdout.txt

In [ ]:
# 7) Result: turn the log into a table, produce a runs.csv row, list the videos
!curl -sO https://raw.githubusercontent.com/mesutd0nmez/vla-robot-learning-tr/main/code/day12/parse_eval_log.py
import glob
logs = sorted(glob.glob("checkpoints/libero_spatial_checkpoint_*/eval.log"))
print(logs)
!python parse_eval_log.py {logs[-1]} --run-id vlanext_spatial_t4_01 --csv results/runs.csv
!ls checkpoints/libero_spatial_checkpoint_*/*.mp4

In [ ]:
# 8) Watch a rollout video inside the notebook (pick a failed one: where does it get stuck?)
from IPython.display import Video
import glob
vids = sorted(glob.glob("checkpoints/libero_spatial_checkpoint_*/*.mp4"))
Video(vids[0], embed=True, width=384)

## Next
- Download `results/runs.csv` and `eval.log` (Files panel) → put them in an experiment folder under `projects/` in the repo.
- In Lecture 13 we will read the inside of this code: where the command enters (`spec.task_description`), how the action is decoded (`_denormalize_action`), where success is computed (`env.step` → `done`).
- The only line to change for the Turkish experiment is `task_description = task.language` inside `build_episode_specs` — but we will do that in Lecture 18.